In [ ]:
import re
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment

try:
    import pymorphy2
    _morph = pymorphy2.MorphAnalyzer()
    _USE_MORPH = True
except ImportError:
    _morph = None
    _USE_MORPH = False
    print(
        "[WARNING] pymorphy2 не установлен. Лемматизация отключена.\n"
        "Установите: pip install pymorphy2 pymorphy2-dicts-ru"
    )

_lemma_cache: dict = {}
_phrase_cache: dict = {}


def lemmatize_word(word: str) -> str:
    if not _USE_MORPH:
        return word
    if word not in _lemma_cache:
        _lemma_cache[word] = _morph.parse(word)[0].normal_form
    return _lemma_cache[word]


def get_lemma_phrase(phrase: str) -> str:
    # кешируем целые фразы, а не только отдельные слова — на больших датасетах разница ощутимая
    if phrase not in _phrase_cache:
        _phrase_cache[phrase] = ' '.join(lemmatize_word(w) for w in phrase.split())
    return _phrase_cache[phrase]


def normalize_text(text) -> str:
    if pd.isna(text):
        return ""
    return re.sub(r'\s+', ' ', str(text).strip().lower())


def is_partial_match(a: str, b: str) -> bool:
    # partial = "одно входит в другое" (леммы), но не для односложных фраз — там только полное совпадение
    la = get_lemma_phrase(a)
    lb = get_lemma_phrase(b)

    if la == lb:
        return True

    a_words = la.split()
    b_words = lb.split()

    # "курс" vs "онлайн-курс" — односложное не дробим, чтобы не размывать точность
    if len(a_words) == 1 and len(b_words) == 1:
        return False

    if len(a_words) == 1:
        return a_words[0] in b_words
    if len(b_words) == 1:
        return b_words[0] in a_words

    return la in lb or lb in la


def partial_quad_match(pred_quad: tuple, gold_quad: tuple) -> bool:
    # квадруплет совпал, если аспект+тональность точно, а термин+мнение частично
    pred_term, pred_op, pred_asp, pred_sent = pred_quad
    gold_term, gold_op, gold_asp, gold_sent = gold_quad

    if pred_asp != gold_asp or pred_sent != gold_sent:
        return False

    return is_partial_match(pred_term, gold_term) and is_partial_match(pred_op, gold_op)


def build_quad_list(rows: pd.DataFrame) -> list:
    quads = []
    for _, row in rows.iterrows():
        term = normalize_text(row['aspect_term'])
        op   = normalize_text(row['opinion'])
        asp  = normalize_text(row['aspect'])
        sent = normalize_text(row['sentiment'])
        quads.append((term, op, asp, sent))
    return quads


def optimal_match(pred_list: list, gold_list: list, match_fn) -> tuple:
    # венгерский алгоритм: каждому предсказанию ищем оптимальную пару из golden,
    # чтобы максимизировать общее число совпадений. на greedy получалось хуже,
    # потому что одна модель могла схватить несколько разных gold-квадруплетов одним предсказанием
    n_pred = len(pred_list)
    n_gold = len(gold_list)

    if n_pred == 0 and n_gold == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gold
    if n_gold == 0:
        return 0, n_pred, 0

    match_matrix = np.zeros((n_pred, n_gold), dtype=int)

    for i, pred in enumerate(pred_list):
        for j, gold in enumerate(gold_list):
            if match_fn(pred, gold):
                match_matrix[i, j] = 1

    # linear_sum_assignment минимизирует cost, поэтому инвертируем знак
    cost_matrix = -match_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    tp = sum(match_matrix[i, j] for i, j in zip(row_ind, col_ind))
    fp = n_pred - tp
    fn = n_gold - tp

    return tp, fp, fn


def compute_prf(tp: int, fp: int, fn: int):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def compute_metrics(golden_df: pd.DataFrame, pred_df: pd.DataFrame, mode: str = 'exact') -> dict:
    golden_df = golden_df.copy()
    pred_df   = pred_df.copy()

    golden_df['review_id'] = golden_df['review_id'].astype(str)
    pred_df['review_id']   = pred_df['review_id'].astype(str)

    golden_grouped = golden_df.groupby('review_id')
    pred_grouped   = pred_df.groupby('review_id')

    counters = {
        'quad': [0, 0, 0],
        'term': [0, 0, 0],
        'op':   [0, 0, 0],
        'asp':  [0, 0, 0],
        'sent': [0, 0, 0],
    }

    def add(key, tp=0, fp=0, fn=0):
        counters[key][0] += tp
        counters[key][1] += fp
        counters[key][2] += fn

    all_review_ids = set(golden_grouped.groups.keys()) | set(pred_grouped.groups.keys())

    for rid in all_review_ids:
        gold_rows = golden_grouped.get_group(rid) if rid in golden_grouped.groups else pd.DataFrame()
        pred_rows = pred_grouped.get_group(rid) if rid in pred_grouped.groups else pd.DataFrame()

        gold_list = build_quad_list(gold_rows)
        pred_list = build_quad_list(pred_rows)

        if mode == 'exact':
            # exact = все четыре поля байт-в-байт после нормализации
            tp, fp, fn = optimal_match(pred_list, gold_list, lambda p, g: p == g)
            add('quad', tp=tp, fp=fp, fn=fn)

            for key, idx in [('term', 0), ('op', 1), ('asp', 2), ('sent', 3)]:
                gold_vals = [q[idx] for q in gold_list]
                pred_vals = [q[idx] for q in pred_list]
                tp, fp, fn = optimal_match(pred_vals, gold_vals, lambda p, g: p == g)
                add(key, tp=tp, fp=fp, fn=fn)

        elif mode == 'partial':
            # partial = термин+мнение через подстроки/леммы, аспект+тональность точно
            tp, fp, fn = optimal_match(pred_list, gold_list, partial_quad_match)
            add('quad', tp=tp, fp=fp, fn=fn)

            for key, idx in [('term', 0), ('op', 1)]:
                gold_vals = [q[idx] for q in gold_list]
                pred_vals = [q[idx] for q in pred_list]
                tp, fp, fn = optimal_match(pred_vals, gold_vals, is_partial_match)
                add(key, tp=tp, fp=fp, fn=fn)

            for key, idx in [('asp', 2), ('sent', 3)]:
                gold_vals = [q[idx] for q in gold_list]
                pred_vals = [q[idx] for q in pred_list]
                tp, fp, fn = optimal_match(pred_vals, gold_vals, lambda p, g: p == g)
                add(key, tp=tp, fp=fp, fn=fn)

    return {
        'Aspect Term': compute_prf(*counters['term']),
        'Opinion':     compute_prf(*counters['op']),
        'Aspect':      compute_prf(*counters['asp']),
        'Sentiment':   compute_prf(*counters['sent']),
        'Quad':        compute_prf(*counters['quad']),
    }


if __name__ == "__main__":
    golden_df = pd.read_csv('golden_dataset_3.csv')
    pred_df   = pd.read_csv('predictions_dataset_gpt.csv')

    print("\n=== Exact Match ===")
    for name, (p, r, f) in compute_metrics(golden_df, pred_df, mode='exact').items():
        print(f"{name:<15} P={p:.4f} R={r:.4f} F1={f:.4f}")

    print("\n=== Partial Match ===")
    for name, (p, r, f) in compute_metrics(golden_df, pred_df, mode='partial').items():
        print(f"{name:<15} P={p:.4f} R={r:.4f} F1={f:.4f}")